# 3.2 Modelo LUR por Barrios (LSOA-level)


Construye un modelo de Regresión de Uso de Suelo (LUR) a nivel de LSOA (Lower layer Super Output Area).  
La variable dependiente es la concentración media de PM2.5 y PM10 por barrio, derivada de los sensores.  
Las variables independientes son características espaciales del entorno urbano (OSMnx).

## Pipeline
1. Agregar lecturas de sensores → media temporal por sensor
2. Descargar límites LSOA de Liverpool (ONS GeoPortal)
3. Asignar cada sensor a su LSOA
4. Calcular estadísticas zonales (landuse, buildings, streets) por LSOA
5. Construir matriz de features
6. Entrenar modelo LUR (OLS + Random Forest)
7. Generar mapa de predicción para todos los LSOAs
8. Diagnósticos y exportación

In [ ]:
# =========================
# IMPORTS
# =========================
import io
import logging
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler
from shapely.geometry import Point

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
# =========================
# CONFIG
# =========================
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_DIR       = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUTS_DIR   = ROOT / "outputs"
FIGURES_DIR   = OUTPUTS_DIR / "figures" / "lur_barrios"
MODELS_DIR    = OUTPUTS_DIR / "models"

for d in [PROCESSED_DIR, FIGURES_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CRS_GEO        = "EPSG:4326"
CRS_PROJECTED  = "EPSG:27700"  # British National Grid — unidades en metros

TARGET_VARS    = ["PM2.5", "PM10"]

# Código LAD de Liverpool (E08000012 — Liverpool UA)
LIVERPOOL_LAD_CODE = "E08000012"

logger.info("ROOT: %s", ROOT)

---
## 1. Carga y Agregación de Sensores

In [ ]:
# === 1. SENSORES ===
logger.info("Cargando sensors_definitivo.csv...")

sensors_raw = pd.read_csv(
    PROCESSED_DIR / "sensors_definitivo.csv",
    parse_dates=["datetime"],
    low_memory=False
)

logger.info("Filas totales: %s | Sensores únicos: %s",
            len(sensors_raw), sensors_raw["sensor_id"].nunique())
sensors_raw.head(3)

In [ ]:
# Filtrar solo lecturas válidas y calcular media temporal por sensor
# Usamos el periodo completo disponible para maximizar representatividad
if "tipo" in sensors_raw.columns:
    sensors_valid = sensors_raw[sensors_raw["tipo"] == "valido"].copy()
else:
    sensors_valid = sensors_raw.copy()

sensors_mean = (
    sensors_valid
    .groupby("sensor_id", as_index=False)
    .agg(
        lat=("lat", "first"),
        lon=("lon", "first"),
        label=("label", "first"),
        **{v: (v, "mean") for v in TARGET_VARS},
        n_obs=("PM2.5", "count"),
    )
)

logger.info("Sensores con media calculada: %d", len(sensors_mean))
sensors_mean[["sensor_id", "label", "lat", "lon"] + TARGET_VARS + ["n_obs"]]

In [ ]:
# Convertir a GeoDataFrame en WGS84 y proyectar a BNG
gdf_sensors = gpd.GeoDataFrame(
    sensors_mean,
    # NOTA: en sensors_definitivo.csv las columnas "lat" y "lon" están intercambiadas
    # "lat" contiene valores de longitud (~-2.98) y "lon" contiene latitud (~53.4)
    geometry=gpd.points_from_xy(sensors_mean["lat"], sensors_mean["lon"]),
    crs=CRS_GEO
).to_crs(CRS_PROJECTED)

logger.info("GeoDataFrame sensores: %d puntos | CRS: %s", len(gdf_sensors), gdf_sensors.crs)
ax = gdf_sensors.plot(markersize=5, color="crimson", figsize=(6, 6))
ax.set_title("Sensores Liverpool")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensores_ubicacion.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 2. Descarga de Límites LSOA Liverpool

In [ ]:
# === 2. LÍMITES LSOA ===
# Fuente: ONS GeoPortal — LSOA Dec 2021, Super Generalised Clipped (BSC V4)
# Estrategia: lookup table (LTLA22CD=E08000012) → batch de geometrías por códigos
LSOA_CACHE = RAW_DIR / "lsoa_liverpool.gpkg"

LIVERPOOL_LAD_CODE = "E08000012"

def download_lsoa_liverpool() -> gpd.GeoDataFrame:
    if LSOA_CACHE.exists():
        gdf = gpd.read_file(LSOA_CACHE)
        # Validar que son LSOAs reales (códigos E01XXXXXX de 9 chars)
        if gdf["LSOA21CD"].str.match(r"E01\d{6}").all():
            logger.info("LSOA cargadas desde caché: %d polígonos", len(gdf))
            return gdf
        logger.warning("Caché contiene LSOAs sintéticas — re-descargando...")
        LSOA_CACHE.unlink()

    logger.info("Descargando LSOA Liverpool desde ONS GeoPortal...")

    # Paso 1: obtener códigos LSOA de Liverpool vía lookup table
    lookup_url = (
        "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_2021_to_Ward_to_Lower_Tier_Local_Authority_May_2022_Lookup_for_England_2022"
        "/FeatureServer/0/query"
        f"?where=LTLA22CD%3D%27{LIVERPOOL_LAD_CODE}%27"
        "&outFields=LSOA21CD"
        "&returnGeometry=false"
        "&f=json"
        "&resultRecordCount=2000"
    )
    resp = requests.get(lookup_url, timeout=30)
    resp.raise_for_status()
    codes = [f["attributes"]["LSOA21CD"] for f in resp.json().get("features", [])]
    if not codes:
        raise RuntimeError("Lookup table returned 0 LSOAs for Liverpool (E08000012).")
    logger.info("Códigos LSOA obtenidos: %d", len(codes))

    # Paso 2: descargar geometrías en lotes de 100
    all_gdfs = []
    batch_size = 100
    for i in range(0, len(codes), batch_size):
        batch = codes[i : i + batch_size]
        code_list = "','".join(batch)
        geom_url = (
            "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
            "LSOA_2021_EW_BSC_V4_RUC/FeatureServer/0/query"
            f"?where=LSOA21CD+IN+('{code_list}')"
            "&outFields=LSOA21CD%2CLSOA21NM"
            "&returnGeometry=true"
            "&outSR=27700"
            "&f=geojson"
            "&resultRecordCount=200"
        )
        r2 = requests.get(geom_url, timeout=60)
        r2.raise_for_status()
        gdf_batch = gpd.read_file(io.BytesIO(r2.content))
        all_gdfs.append(gdf_batch)
        logger.info("  Lote %d/%d: %d features", i // batch_size + 1,
                    -(-len(codes) // batch_size), len(gdf_batch))

    gdf = gpd.pd.concat(all_gdfs, ignore_index=True)
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:27700")

    gdf.to_file(LSOA_CACHE, driver="GPKG")
    logger.info("LSOA guardadas: %d polígonos → %s", len(gdf), LSOA_CACHE)
    return gdf


gdf_lsoa = download_lsoa_liverpool()
logger.info("LSOAs Liverpool: %d | CRS: %s", len(gdf_lsoa), gdf_lsoa.crs)
gdf_lsoa.head(3)

In [ ]:
# Calcular área en km² para normalización posterior
gdf_lsoa["area_km2"] = gdf_lsoa.geometry.area / 1e6

fig, ax = plt.subplots(figsize=(7, 8))
gdf_lsoa.plot(ax=ax, facecolor="lightyellow", edgecolor="gray", linewidth=0.4)
gdf_sensors.plot(ax=ax, color="crimson", markersize=8, zorder=5, label="Sensores")
ax.set_title("LSOAs Liverpool con sensores", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lsoa_sensores.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 3. Asignación Sensores → LSOA

In [ ]:
# === 3. SPATIAL JOIN SENSORES → LSOA ===
gdf_joined = gpd.sjoin(
    gdf_sensors,
    gdf_lsoa[["LSOA21CD", "LSOA21NM", "geometry"]],
    how="left",
    predicate="within"
)

# Sensores fuera de cualquier LSOA (borde o exteriores)
fuera = gdf_joined["LSOA21CD"].isna()
if fuera.any():
    logger.warning("%d sensor(es) fuera de LSOAs — se asignan por nearest.", fuera.sum())
    nearest = gpd.sjoin_nearest(
        gdf_sensors[fuera],
        gdf_lsoa[["LSOA21CD", "LSOA21NM", "geometry"]],
        how="left"
    )[["sensor_id", "LSOA21CD", "LSOA21NM"]]
    gdf_joined.loc[fuera, "LSOA21CD"] = nearest["LSOA21CD"].values
    gdf_joined.loc[fuera, "LSOA21NM"] = nearest["LSOA21NM"].values

# Si hay varios sensores en la misma LSOA → media ponderada por n_obs
def weighted_mean(df, col, weight_col="n_obs"):
    return np.average(df[col], weights=df[weight_col])

sensors_per_lsoa = (
    gdf_joined
    .groupby("LSOA21CD")
    .apply(lambda g: pd.Series({
        "PM2.5_mean": weighted_mean(g, "PM2.5"),
        "PM10_mean":  weighted_mean(g, "PM10"),
        "n_sensores": len(g),
        "n_obs_total": g["n_obs"].sum(),
    }))
    .reset_index()
)

logger.info("LSOAs con al menos un sensor: %d / %d", len(sensors_per_lsoa), len(gdf_lsoa))
sensors_per_lsoa

---
## 4. Estadísticas Zonales (OSMnx)

> **Prerequisito:** ejecutar `python src/analysis/extract_osm_features.py` para generar los GPKGs en `data/raw/`.

In [ ]:
# === 4. CARGA CAPAS OSMnx ===
REQUIRED_GPKGS = {
    "landuse":   RAW_DIR / "landuse_liverpool.gpkg",
    "buildings": RAW_DIR / "buildings_liverpool.gpkg",
    "streets":   RAW_DIR / "streets_liverpool.gpkg",
}

missing = [k for k, p in REQUIRED_GPKGS.items() if not p.exists()]
if missing:
    raise FileNotFoundError(
        f"Faltan GPKGs: {missing}. "
        "Ejecuta primero: python src/analysis/extract_osm_features.py"
    )

gdf_landuse   = gpd.read_file(REQUIRED_GPKGS["landuse"])
gdf_buildings = gpd.read_file(REQUIRED_GPKGS["buildings"])
gdf_streets   = gpd.read_file(REQUIRED_GPKGS["streets"])

for name, gdf in [("landuse", gdf_landuse), ("buildings", gdf_buildings), ("streets", gdf_streets)]:
    logger.info("%s: %d filas | CRS: %s", name, len(gdf), gdf.crs)

In [ ]:
# === 4a. ESTADÍSTICAS ZONALES — LANDUSE ===
# Para cada LSOA: % de área ocupada por cada tipo de uso de suelo

LANDUSE_TYPES = {
    "industrial":  ["industrial"],
    "commercial":  ["commercial"],
    "residential": ["residential"],
    "green":       ["park", "garden", "grass", "forest"],
}

col_lu = "landuse" if "landuse" in gdf_landuse.columns else "leisure"

def compute_landuse_stats(lsoa: gpd.GeoDataFrame, lu: gpd.GeoDataFrame) -> pd.DataFrame:
    records = []
    for _, row in lsoa.iterrows():
        lsoa_geom = row.geometry
        lsoa_area = lsoa_geom.area
        rec = {"LSOA21CD": row["LSOA21CD"]}

        # Clip landuse a esta LSOA
        clip = lu[lu.geometry.intersects(lsoa_geom)].copy()
        clip["geom_clip"] = clip.geometry.intersection(lsoa_geom)

        for feat, categories in LANDUSE_TYPES.items():
            mask = clip[col_lu].isin(categories) if col_lu in clip.columns else pd.Series(False, index=clip.index)
            area = clip.loc[mask, "geom_clip"].area.sum()
            rec[f"pct_{feat}"] = (area / lsoa_area) * 100 if lsoa_area > 0 else 0.0

        records.append(rec)
    return pd.DataFrame(records)


logger.info("Calculando estadísticas de landuse por LSOA (puede tardar unos minutos)...")
df_landuse_stats = compute_landuse_stats(gdf_lsoa, gdf_landuse)
logger.info("Landuse stats: %s", df_landuse_stats.shape)
df_landuse_stats.head(3)

In [ ]:
# === 4b. ESTADÍSTICAS ZONALES — BUILDINGS ===
# Building Coverage Ratio (BCR): área edificada / área LSOA

def compute_building_stats(lsoa: gpd.GeoDataFrame, bldg: gpd.GeoDataFrame) -> pd.DataFrame:
    records = []
    for _, row in lsoa.iterrows():
        lsoa_geom = row.geometry
        lsoa_area = lsoa_geom.area
        clip = bldg[bldg.geometry.intersects(lsoa_geom)].copy()
        clip["geom_clip"] = clip.geometry.intersection(lsoa_geom)

        built_area = clip["geom_clip"].area.sum()
        records.append({
            "LSOA21CD":         row["LSOA21CD"],
            "bcr":              (built_area / lsoa_area) * 100 if lsoa_area > 0 else 0.0,
            "n_buildings":      len(clip),
            "mean_bldg_area":   clip["geom_clip"].area.mean() if len(clip) > 0 else 0.0,
        })
    return pd.DataFrame(records)


logger.info("Calculando estadísticas de edificios por LSOA...")
df_building_stats = compute_building_stats(gdf_lsoa, gdf_buildings)
logger.info("Building stats: %s", df_building_stats.shape)
df_building_stats.head(3)

In [ ]:
# === 4c. ESTADÍSTICAS ZONALES — STREETS ===
# Densidad de carreteras por tipo (m/km²) dentro de cada LSOA

HIGHWAY_TYPES = ["motorway", "primary", "secondary", "residential"]
col_hw = "highway" if "highway" in gdf_streets.columns else None

def compute_street_stats(lsoa: gpd.GeoDataFrame, streets: gpd.GeoDataFrame) -> pd.DataFrame:
    records = []
    for _, row in lsoa.iterrows():
        lsoa_geom = row.geometry
        lsoa_area_km2 = lsoa_geom.area / 1e6
        clip = streets[streets.geometry.intersects(lsoa_geom)].copy()
        clip["geom_clip"] = clip.geometry.intersection(lsoa_geom)

        rec = {"LSOA21CD": row["LSOA21CD"]}

        if col_hw and col_hw in clip.columns:
            for hw in HIGHWAY_TYPES:
                mask = clip[col_hw] == hw
                length_m = clip.loc[mask, "geom_clip"].length.sum()
                rec[f"street_density_{hw}"] = length_m / lsoa_area_km2 if lsoa_area_km2 > 0 else 0.0

        total_length = clip["geom_clip"].length.sum()
        rec["street_density_total"] = total_length / lsoa_area_km2 if lsoa_area_km2 > 0 else 0.0
        records.append(rec)
    return pd.DataFrame(records)


logger.info("Calculando densidad de calles por LSOA...")
df_street_stats = compute_street_stats(gdf_lsoa, gdf_streets)
logger.info("Street stats: %s", df_street_stats.shape)
df_street_stats.head(3)

---
## 5. Construcción de la Matriz de Features

In [ ]:
# === 5. FEATURE MATRIX ===
# Unir todas las capas al GeoDataFrame de LSOAs
gdf_features = (
    gdf_lsoa[["LSOA21CD", "LSOA21NM", "area_km2", "geometry"]]
    .merge(df_landuse_stats,   on="LSOA21CD", how="left")
    .merge(df_building_stats,  on="LSOA21CD", how="left")
    .merge(df_street_stats,    on="LSOA21CD", how="left")
)

# Unir las medias de los sensores (solo LSOAs con datos reales)
gdf_model = gdf_features.merge(sensors_per_lsoa, on="LSOA21CD", how="left")

FEATURE_COLS = (
    [f"pct_{k}" for k in LANDUSE_TYPES] +
    ["bcr", "n_buildings", "mean_bldg_area"] +
    [f"street_density_{hw}" for hw in HIGHWAY_TYPES] +
    ["street_density_total"]
)
# Solo columnas que existen
FEATURE_COLS = [c for c in FEATURE_COLS if c in gdf_model.columns]

logger.info("Features disponibles (%d): %s", len(FEATURE_COLS), FEATURE_COLS)

# Rellenar NaN en features con 0 (LSOA sin intersección con esa capa)
gdf_model[FEATURE_COLS] = gdf_model[FEATURE_COLS].fillna(0)

logger.info("LSOAs totales: %d | Con datos de sensor: %d",
            len(gdf_model), gdf_model["PM2.5_mean"].notna().sum())
gdf_model.head(3)

In [ ]:
# Correlación features vs PM2.5 (solo LSOAs con sensor)
df_train = gdf_model[gdf_model["PM2.5_mean"].notna()].copy()

corr = df_train[FEATURE_COLS + ["PM2.5_mean", "PM10_mean"]].corr()[["PM2.5_mean", "PM10_mean"]]
corr_sorted = corr.drop(["PM2.5_mean", "PM10_mean"]).sort_values("PM2.5_mean", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
corr_sorted["PM2.5_mean"].plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlación Pearson: Features vs PM2.5 (LSOAs con sensor)")
ax.set_xlabel("r de Pearson")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlacion_features_pm25.png", dpi=150, bbox_inches="tight")
plt.show()

corr_sorted

---
## 6. Modelo LUR

Se ajustan dos modelos:
- **OLS** (statsmodels) → interpretabilidad, p-values por variable
- **Random Forest** → captura no-linealidades, feature importance

Validación: **Leave-One-Out Cross-Validation (LOOCV)** sobre las LSOAs con sensor.

In [ ]:
# === 6. MODELO LUR ===

def evaluate_loocv(X: np.ndarray, y: np.ndarray, model) -> dict:
    """Leave-One-Out CV. Devuelve R², RMSE y predicciones OOF."""
    if len(y) < 2:
        raise ValueError(f"LOOCV requiere al menos 2 muestras; se tienen {len(y)}.")
    loo = LeaveOneOut()
    y_pred = np.zeros_like(y, dtype=float)
    for train_idx, test_idx in loo.split(X):
        model.fit(X[train_idx], y[train_idx])
        y_pred[test_idx] = model.predict(X[test_idx])
    r2   = r2_score(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    return {"r2": r2, "rmse": rmse, "y_pred_oof": y_pred}


results = {}

for target in TARGET_VARS:
    target_col = f"{target}_mean"
    df_t = df_train[FEATURE_COLS + [target_col]].dropna()
    X = df_t[FEATURE_COLS].values
    y = df_t[target_col].values

    logger.info("--- %s | n=%d ---", target, len(y))

    scaler = StandardScaler()
    X_sc   = scaler.fit_transform(X)

    # --- OLS con statsmodels (fit completo para p-values y summary) ---
    ols_full = sm.OLS(y, sm.add_constant(X_sc)).fit()

    # --- LOOCV para OLS usando sklearn LinearRegression ---
    from sklearn.linear_model import LinearRegression as LR
    ols_cv = evaluate_loocv(X_sc, y, LR())

    # --- Random Forest con LOOCV ---
    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf_cv = evaluate_loocv(X, y, rf)

    # Ajustar RF completo para feature importance y predicción posterior
    rf.fit(X, y)

    results[target] = {
        "ols_summary":  ols_full.summary(),
        "ols_cv":       ols_cv,
        "rf_cv":        rf_cv,
        "rf_model":     rf,
        "scaler":       scaler,
        "feature_cols": FEATURE_COLS,
        "X": X, "y": y,
        "df_idx": df_t.index,
    }

    logger.info("%s | OLS LOOCV  → R²=%.3f | RMSE=%.3f", target, ols_cv["r2"], ols_cv["rmse"])
    logger.info("%s | RF  LOOCV  → R²=%.3f | RMSE=%.3f", target, rf_cv["r2"],  rf_cv["rmse"])

In [ ]:
# OLS Summary para PM2.5
print(results["PM2.5"]["ols_summary"])

In [ ]:
# Feature Importance (Random Forest)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, target in zip(axes, TARGET_VARS):
    rf_model = results[target]["rf_model"]
    importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
    importances.plot(kind="barh", ax=ax, color="teal")
    ax.set_title(f"RF Feature Importance — {target}")
    ax.set_xlabel("Importancia")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "feature_importance_rf.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Gráficos observado vs predicho (LOOCV)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for row_i, target in enumerate(TARGET_VARS):
    y = results[target]["y"]
    for col_i, (model_name, cv_key) in enumerate([("OLS", "ols_cv"), ("Random Forest", "rf_cv")]):
        ax = axes[row_i][col_i]
        y_pred = results[target][cv_key]["y_pred_oof"]
        r2     = results[target][cv_key]["r2"]
        rmse   = results[target][cv_key]["rmse"]

        ax.scatter(y, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5)
        lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
        ax.plot(lims, lims, "r--", linewidth=1)
        ax.set_xlabel(f"{target} observado (μg/m³)")
        ax.set_ylabel(f"{target} predicho (μg/m³)")
        ax.set_title(f"{model_name} — {target}\nR²={r2:.3f} | RMSE={rmse:.3f}")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "obs_vs_pred_loocv.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 7. Predicción para Todos los LSOAs

In [ ]:
# === 7. PREDICCIÓN COMPLETA ===
# Usamos el mejor modelo (RF) para predecir en todos los LSOAs
X_all = gdf_model[FEATURE_COLS].fillna(0).values

for target in TARGET_VARS:
    rf_model = results[target]["rf_model"]
    gdf_model[f"{target}_pred"] = rf_model.predict(X_all)

# Donde hay sensor: mostrar valor real; resto: predicción
for target in TARGET_VARS:
    gdf_model[f"{target}_final"] = gdf_model[f"{target}_mean"].fillna(gdf_model[f"{target}_pred"])

logger.info("Predicción completada para %d LSOAs", len(gdf_model))

In [ ]:
# Mapa de predicción PM2.5
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, target in zip(axes, TARGET_VARS):
    col = f"{target}_final"
    gdf_model.plot(
        column=col,
        cmap="YlOrRd",
        legend=True,
        legend_kwds={"label": f"{target} (μg/m³)", "shrink": 0.6},
        ax=ax,
        edgecolor="gray",
        linewidth=0.2,
        missing_kwds={"color": "lightgrey", "label": "Sin datos"}
    )
    # Marcar sensores reales
    gdf_sensors.plot(ax=ax, color="blue", markersize=6, zorder=5, label="Sensor real")
    ax.set_title(f"Modelo LUR — {target} por LSOA", fontsize=12)
    ax.legend(fontsize=8)
    ax.axis("off")

plt.suptitle("Mapa de Contaminación Estimada — Liverpool", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "mapa_prediccion_lsoa.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 8. Exportación de Resultados

In [ ]:
# === 8. EXPORTACIÓN ===
# GeoJSON con predicciones por LSOA (entregable Issue #21)
export_cols = (
    ["LSOA21CD", "LSOA21NM", "area_km2"]
    + FEATURE_COLS
    + [f"{t}_mean"  for t in TARGET_VARS]
    + [f"{t}_pred"  for t in TARGET_VARS]
    + [f"{t}_final" for t in TARGET_VARS]
    + ["n_sensores", "geometry"]
)
export_cols = [c for c in export_cols if c in gdf_model.columns]

out_geojson = PROCESSED_DIR / "lur_barrios_predictions.geojson"
gdf_model[export_cols].to_crs(CRS_GEO).to_file(out_geojson, driver="GeoJSON")
logger.info("Exportado: %s", out_geojson)

# CSV plano sin geometría
out_csv = PROCESSED_DIR / "lur_barrios_predictions.csv"
gdf_model[export_cols].drop(columns="geometry").to_csv(out_csv, index=False)
logger.info("Exportado: %s", out_csv)

# Resumen métricas
print("\n=== MÉTRICAS LOOCV ===")
for target in TARGET_VARS:
    for model_name, cv_key in [("OLS", "ols_cv"), ("RF", "rf_cv")]:
        r2   = results[target][cv_key]["r2"]
        rmse = results[target][cv_key]["rmse"]
        print(f"  {target} | {model_name:12s} → R²={r2:.3f} | RMSE={rmse:.3f} μg/m³")